# Exploratory Synthetic Data Analysis - Google Colab

## 1. Secure Credential Injection

Do **not hardcode** your Hugging Face access token (HF_TOKEN) inside a Colab code cell. 

Instead, leverage Colab's native **Secrets management system** to inject credentials securely into the environment execution plane.

Click on the **Key icon** (Secrets) located in the left-hand sidebar of the Colab interface.

Add a new secret with the following key-value layout:

Name: HF_TOKEN

Value: [Your Hugging Face Write Token beginning with hf_]

Toggle the Notebook access switch to On for that specific secret.

## Dependencies and Authentication Boilerplate

In [ ]:
# 1. Install required framework dependencies silently
!pip install -q --upgrade huggingface_hub datasets transformers TRL accelerate

import os
from google.colab import userdata
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import pandas as pd

# 2. Inject the secure secret into the OS environment block
# This must happen BEFORE invoking Hugging Face APIs
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

print("[*] Environment authenticated successfully.")

## Pulling Data & Models

In [ ]:
# Define target repository identifiers
BASE_DATASET_ID = "Jacaranda-Dev/waf-dataset-base"            # base splits from public datasets  (deduped)
SYNTHESIS_DATASET_ID = "Jacaranda-Dev/waf-dataset-synthesis"  # Attack payloads only (Grammar + LLM)
FRAMED_DATASET_ID = "Jacaranda-Dev/waf-dataset-framed"        # Attack payloads and innocent payloads wrapped inside requests
AUGMENTED_DATASET_ID = "Jacaranda-Dev/waf-dataset-augmented"  # Final augmented splits

TOKENIZER_MODEL_ID = "Jacaranda-Dev/waf-tokenizer-track-b"
TEACHER_MODEL_ID = "Jacaranda-Dev/Jacaranda-Dev/waf-teacher-99m"
STUDENT_MODEL_ID = "Jacaranda-Dev/Jacaranda-Dev/waf-student-10m"



# Pull the dataset into memory/local disk cache
#print("[*] Pulling dataset from Hub...")
#base_dataset = load_dataset(BASE_DATASET_ID)
#synthesis_dataset = load_dataset(SYNTHESIS_DATASET_ID)
#framed_dataset = load_dataset(FRAMED_DATASET_ID)
#augmented_dataset = load_dataset(AUGMENTED_DATASET_ID)

# Instantiate the model weights and tokenizer
#print("[*] Pulling model architectural weights...")
#tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_MODEL_ID)
#teacher_model = AutoModelForCausalLM.from_pretrained(TEACHER_MODEL_ID, device_map="auto")
#student_model = AutoModelForCausalLM.from_pretrained(STUDENT_MODEL_ID, device_map="auto")

#### Inspection of a parquet file

In [ ]:

# 1. Load the dataset from the Hugging Face Hub
print("[*] Pulling dataset from Hub...")
dataset_dict = load_dataset(BASE_DATASET_ID)
#dataset_dict = load_dataset(SYNTHESIS_DATASET_ID)
#dataset_dict = load_dataset(FRAMED_DATASET_ID)
#dataset_dict = load_dataset(AUGMENTED_DATASET_ID)

# 2. Extract a specific split (e.g., 'train') and convert to a Pandas DataFrame
df = dataset_dict['train'].to_pandas()
#df = dataset_dict['val'].to_pandas()
#df = dataset_dict['test'].to_pandas()
#df = dataset_dict['adversarial'].to_pandas()
#df = dataset_dict['canary'].to_pandas()

print(df.shape)
print(df.head())

# --- Your exact inspection script now runs natively below ---
print(f"--- Augmented Dataset ---")
print("Columns:", df.columns.tolist())

print("\n--- Rows ---")
print(f"{len(df)}")

print("\n--- Unique Categories ---")
print(df['attack_class'].unique())

print("\n--- Unique Labels ---")
print(df['label'].unique())

print("\n--- Distribution (Counts) ---")
print(df.groupby(['attack_class', 'label']).size().reset_index(name='count'))

with pd.option_context('display.max_colwidth', None):
    # See 5 random rows to check different attack categories
    result = df.sample(5)
    display(result)

print("\n--- Unique Raw ---")
print(df['raw'].nunique())

--- Augmented Dataset ---
Columns: ['id', 'method', 'path', 'query_string', 'headers', 'body', 'raw', 'label', 'attack_class', 'source', 'split']

--- Rows ---
39506

--- Unique Categories ---
['xxe' 'header_injection' 'ssti' 'ssrf' 'xss' 'lfi' 'cmdi'
 'path_traversal']

--- Unique Labels ---
[1]

--- Distribution (Counts) ---
       attack_class  label  count
0              cmdi      1   4911
1  header_injection      1   5000
2               lfi      1   5000
3    path_traversal      1   5000
4              ssrf      1   5000
5              ssti      1   5000
6               xss      1   4595
7               xxe      1   5000


,id,method,path,query_string,headers,body,raw,label,attack_class,source,split
2630,20376daa-ab65-434b-9d74-6dbc039524dd,POST,/api/v1/xml,,"{""Host"": ""stub.invalid"", ""User-Agent"": ""stub""}",```xml\n<!DOCTYPE%0droot%09[%09<!ENTITY%0d%\txxe%0dSYSTEM 'http://internal-ip:8080/sensitive-data.txt'\n>\n]>\n<root><%xxe;></root>,POST /api/v1/xml HTTP/1.1\r\nHost: stub.invalid\r\nUser-Agent: stub\r\n\r\n```xml\n<!DOCTYPE%0droot%09[%09<!ENTITY%0d%\txxe%0dSYSTEM 'http://internal-ip:8080/sensitive-data.txt'\n>\n]>\n<root><%xxe;></root>,1,xxe,aug_synthesis_xxe,
11961,e9897b49-c228-4ded-a992-4db2df728c40,GET,/api/v1/render,template={{7*7}}}},"{""Host"": ""stub.invalid"", ""User-Agent"": ""stub""}",,GET /api/v1/render?template={{7*7}}}} HTTP/1.1\r\nHost: stub.invalid\r\nUser-Agent: stub\r\n,1,ssti,aug_synthesis_ssti,
36544,e8ae7942-1cf6-44e9-9311-8b5c3ca635ff,GET,/download,file=..%2f../../../proc/self/environ%20,"{""Host"": ""stub.invalid"", ""User-Agent"": ""stub""}",,GET /download?file=..%2f../../../proc/self/environ%20 HTTP/1.1\r\nHost: stub.invalid\r\nUser-Agent: stub\r\n,1,path_traversal,aug_synthesis_path_traversal,
4825,4d2b9833-ecc4-4d00-82c7-fb187923c839,POST,/api/v1/xml,,"{""Host"": ""stub.invalid"", ""User-Agent"": ""stub""}",%253C%2521DOCTYPE%2520test%2520SYSTEM%2520%2527xentity%253A%2528http%253A%252F%252Flocalhost%252Fconf%252Fpasswd%2529%2527%253E,POST /api/v1/xml HTTP/1.1\r\nHost: stub.invalid\r\nUser-Agent: stub\r\n\r\n%253C%2521DOCTYPE%2520test%2520SYSTEM%2520%2527xentity%253A%2528http%253A%252F%252Flocalhost%252Fconf%252Fpasswd%2529%2527%253E,1,xxe,aug_synthesis_xxe,
3045,da8dd32f-d0d9-4edb-a737-5cf2cbac16d4,POST,/api/v1/xml,,"{""Host"": ""stub.invalid"", ""User-Agent"": ""stub""}","""```&#120;&#109;&#108;\&#110;<!&#68;&#79;&#67;&#84;&#89;&#80;&#69;/**/&#114;&#111;&#111;&#116;/**/[","POST /api/v1/xml HTTP/1.1\r\nHost: stub.invalid\r\nUser-Agent: stub\r\n\r\n""```&#120;&#109;&#108;\&#110;<!&#68;&#79;&#67;&#84;&#89;&#80;&#69;/**/&#114;&#111;&#111;&#116;/**/[",1,xxe,aug_synthesis_xxe,



--- Unique Raw ---
34219


#### Check Label and Category Ratios

This identifies if your dataset is imbalanced. In WAF research, an **extreme imbalance** (e.g., 99% benign) requires specific handling during training.

In [ ]:

# 1. Binary Label Distribution (0 vs 1)
print("--- Binary Label Counts ---")
print(df['label'].value_counts())
print("\n--- Binary Label Ratios ---")
print(df['label'].value_counts(normalize=True))

# 2. Specific Attack Category Distribution
print("\n--- Attack Category Counts ---")
print(df['attack_class'].value_counts())
print("\n--- Attack Category Ratios ---")
print(df['attack_class'].value_counts(normalize=True))


--- Binary Label Counts ---
label
000 - Normal                              5796
310 - Scanning for Vulnerable Software    2382
126 - Path Traversal                      1806
272 - Protocol Manipulation                 10
66 - SQL Injection                           5
242 - Code Injection                         1
Name: count, dtype: int64

--- Binary Label Ratios ---
label
000 - Normal                              0.5796
310 - Scanning for Vulnerable Software    0.2382
126 - Path Traversal                      0.1806
272 - Protocol Manipulation               0.0010
66 - SQL Injection                        0.0005
242 - Code Injection                      0.0001
Name: proportion, dtype: float64

--- Attack Category Counts ---
category
Normal                              5796
Scanning for Vulnerable Software    2382
Manipulation                        1816
Injection                              6
Name: count, dtype: int64

--- Attack Category Ratios ---
category
Normal                  

#### Cross-Tabulate Label vs. Category

It is a good practice to verify that the label (0 or 1) correctly aligns with the category. This ensures that all categories labeled as "Normal" are actually 0.

In [ ]:
# Create a cross-tabulation table
ctab = pd.crosstab(df['attack_class'], df['label'])
print("--- Label Alignment Check ---")
print(ctab)

--- Label Alignment Check ---
label                             000 - Normal  126 - Path Traversal  \
category                                                               
Injection                                    0                     0   
Manipulation                                 0                  1806   
Normal                                    5796                     0   
Scanning for Vulnerable Software             0                     0   

label                             242 - Code Injection  \
category                                                 
Injection                                            1   
Manipulation                                         0   
Normal                                               0   
Scanning for Vulnerable Software                     0   

label                             272 - Protocol Manipulation  \
category                                                        
Injection                                                 

#### Visualizing with a Bar Chart

In [1]:
import matplotlib.pyplot as plt

# Plotting the categories
plt.figure(figsize=(10, 6))
df['attack_class'].value_counts().plot(kind='bar')
plt.title('Distribution of Attack Categories')
plt.xlabel('Category')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

NameError: name 'df' is not defined

<Figure size 1000x600 with 0 Axes>

#### Payload Length Analysis

Since we are using LLMs/SMLs, the length of the text field matters. If payloads are extremely long, they will be truncated, potentially losing the malicious part of the string.

In [ ]:
# Add a length column for analysis
df['payload_len'] = df['raw'].str.len()

print("--- Payload Length Statistics ---")
print(df.groupby('attack_class')['payload_len'].describe())

# Check for empty payloads
print(f"\nEmpty payloads: {df['raw'].isna().sum()}")

## Using Models

### Loading models

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_MODEL_ID)
teacher_model = AutoModelForSequenceClassification.from_pretrained(TEACHER_MODEL_ID)
student_model = AutoModelForSequenceClassification.from_pretrained(STUDENT_MODEL_ID)


### Running Inference on a request

In [ ]:
import torch
request = "GET /login?user=admin' OR 1=1--"
print('Request:', request)

inputs = tokenizer(request, return_tensors='pt', truncation=True, max_length=512)

outputs = teacher_model(**inputs)
prediction = torch.argmax(outputs.logits, dim=1).item()

print('Prediction:', 'MALICIOUS' if prediction ==1 else 'BENIGN')
